# Chapter 5 — Searching for overdensities, and calibrating the search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/05_overdensity_search.ipynb)

**Goal.** Turn the matched filter (Chapter 4) into a search: scan positions, angular sizes and distances for
compact excesses of satellite-like stars, in the spirit of the SDSS searches of Koposov et al. (2008,
https://arxiv.org/abs/0706.2687) and Walsh et al. (2009, https://doi.org/10.1088/0004-6256/137/1/450). Then ask how
large the maximum of the search statistic is when there is *no* satellite: a search over many trials always
finds something.

**You will learn**
1. to make a significance map from a weighted star map with a small (signal) and a large (background) kernel;
2. why several kernel sizes and distance moduli must be scanned;
3. the **look-elsewhere effect**: the distribution of the *maximum* statistic over the whole search, from
   simulated null patches;
4. to recover three SDSS satellites — Boötes I, Willman 1 and Segue 1 — and judge them against that null.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy import special, stats

DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release once, check its SHA-256, return a DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    assert sha == MANIFEST[name]['sha256'], f'{path} does not match the manifest: delete it and run again'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


lvdb = load('lvdb').set_index('key')
iso = load('isochrones_sdss')

## 5.1 The search statistic

**A CMD mask** (Chapter 4). For a trial distance modulus $\mu$, let $s$ be the CMD density of an old (12.6 Gyr),
metal-poor ([Fe/H] = −2.1) satellite blurred by the errors, and $b$ the CMD density of all stars in the patch (a
satellite is a negligible fraction of them). We rank CMD cells by $s/b$ and keep the top cells that maximise
$\sum s / \sqrt{\sum b}$ (signal over the square root of the background admitted), under the condition that the mask
keeps at least half of the satellite's stars. Stars in the mask get $w = 1$, others $w = 0$.

Why not the continuous weights $s/b$ of Chapter 4? We tried: with them, the statistic below is dominated by a
handful of high-weight stars, its distribution is far from Gaussian, and null patches (Section 5.3) produce
"detections" as strong as real satellites. Counts in a mask are Poisson, and Poisson tails can be computed exactly.

**Maps.** On a grid of 1′ pixels we count masked stars, $W$. A Gaussian kernel $K_\sigma$ (unit sum) gives a local mean
$L = K_\sigma * W$; a broad kernel ($\sigma_{\rm bg} = 30′$) gives the background $\bar W = K_{\rm bg} * W$. The kernel
collects stars from an effective area $A = 1/\sum K_\sigma^2 = 4\pi\sigma^2$ pixels, so $n = A L$ is an effective count
and $\lambda = A \bar W$ its expectation. The significance is the Gaussian-equivalent of the Poisson upper tail,

$$ S = \Phi^{-1}\big[1 - P(N \ge n \mid \lambda)\big], \qquad P(N \ge n \mid \lambda) = \frac{\gamma(n, \lambda)}{\Gamma(n)}, $$

the regularised lower incomplete gamma function, which interpolates the Poisson tail to non-integer $n$.

Convolutions near the edge of the observed area are normalised by the same convolution of the coverage mask.
We compute $S$ for kernels $\sigma = 1′, 2′, 4′, 8′$ and for distance moduli $\mu = 16.5, 17, \ldots, 21.5$, and take
the maximum.

In [ ]:
cb, mb = np.arange(-0.4, 1.21, 0.04), np.arange(16, 22.51, 0.1)
cc, mm = 0.5 * (cb[1:] + cb[:-1]), 0.5 * (mb[1:] + mb[:-1])
CC, MM = np.meshgrid(cc, mm, indexing='ij')
I = iso[np.isclose(iso.logage, 10.1) & np.isclose(iso.feh, -2.1) & (iso.stage <= 7)].sort_values('m_ini')
_c, _g = (I.g - I.r).to_numpy(), I.g.to_numpy()
_arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(_c), np.diff(_g)))])
_s = np.linspace(0, _arc[-1], 3000)
ISO_C, ISO_G, ISO_W = np.interp(_s, _arc, _c), np.interp(_s, _arc, _g), np.gradient(np.interp(_s, _arc, I.int_imf))
MUS = np.arange(16.5, 21.51, 0.5)
SIGMAS = np.array([1, 2, 4, 8])                                  # kernel sizes in arcmin (= pixels)


def satellite_cmd(mu, err=lambda m: 0.02 + 0.08 * np.exp((m - 22) / 1.0)):
    """Normalised satellite CMD density on the (CC, MM) grid for distance modulus mu (as in Chapter 4)."""
    pg = ISO_G + mu
    sc, sm = np.hypot(np.sqrt(2) * err(pg), 0.03), np.hypot(err(pg), 0.1)
    S = np.einsum('k,kij->ij', ISO_W, np.exp(-0.5 * ((CC - ISO_C[:, None, None]) / sc[:, None, None])**2
                                             - 0.5 * ((MM - pg[:, None, None]) / sm[:, None, None])**2) / sc[:, None, None] / sm[:, None, None])
    return S / S.sum()


def cmd_index(c, m):
    i, j = np.digitize(c, cb) - 1, np.digitize(m, mb) - 1
    ok = (i >= 0) & (i < len(cc)) & (j >= 0) & (j < len(mm))
    return np.where(ok, i * len(mm) + j, -1)


def cmd_mask(S, B, keep=0.5):
    """Boolean CMD mask: cells ranked by S/B, the top ones that maximise sum(S)/sqrt(sum(B)) with sum(S) >= keep."""
    order = np.argsort((S / B).ravel())[::-1]
    cs, cbk = np.cumsum(S.ravel()[order]), np.cumsum(B.ravel()[order])
    score = np.where(cs >= keep, cs / np.sqrt(cbk), 0)
    M = np.zeros(S.size, bool)
    M[order[:np.argmax(score) + 1]] = True
    return M.reshape(S.shape)


def cmd_weights(c, m):
    """Mask membership (0/1) of every star for every trial distance modulus: array (len(MUS), n_stars)."""
    k = cmd_index(c, m)
    H = gaussian_filter(np.histogram2d(c, m, [cb, mb])[0], 1.0)
    B = np.maximum(H, 1.0) / np.maximum(H, 1.0).sum()            # field CMD density, floored (Chapter 4)
    return np.array([np.where(k >= 0, cmd_mask(satellite_cmd(mu), B).ravel()[np.maximum(k, 0)], False)
                     for mu in MUS]).astype(float)


def search(x, y, w, mask, extent, pix=1 / 60, sig_bg=30):
    """Significance maps for every (distance modulus, kernel). x, y in deg; w: (n_mu, n_star) weights;
    mask: boolean image of observed pixels on the grid defined by extent = (x0, x1, y0, y1).
    Returns S with shape (n_mu, n_sigma, nx, ny), NaN outside the mask."""
    xe, ye = np.arange(extent[0], extent[1] + pix / 2, pix), np.arange(extent[2], extent[3] + pix / 2, pix)
    M = mask.astype(float)
    norm_bg = gaussian_filter(M, sig_bg)
    out = np.full((len(w), len(SIGMAS)) + mask.shape, np.nan)
    for a, wa in enumerate(w):
        W = np.histogram2d(x, y, [xe, ye], weights=wa)[0]
        Wb = gaussian_filter(W, sig_bg) / np.maximum(norm_bg, 1e-9)
        for b, s in enumerate(SIGMAS):
            norm = gaussian_filter(M, s)
            area = 4 * np.pi * s**2 * norm                         # effective area in pixels, edge-corrected
            n = gaussian_filter(W, s) / np.maximum(norm, 1e-9) * area
            lam = np.maximum(Wb * area, 1e-9)
            p = special.gammainc(np.maximum(n, 1e-9), lam)        # P(N >= n | lam), continuous in n
            S = stats.norm.isf(np.clip(p, 1e-300, 1))
            out[a, b] = np.where(mask & (norm > 0.5) & (n > lam), S, np.where(mask & (norm > 0.5), 0.0, np.nan))
    return out


def peak(S, extent, pix=1 / 60, inner=None):
    """Largest S (optionally only where `inner` is True) and its (mu, sigma, x, y)."""
    T = np.where(inner[None, None] if inner is not None else True, S, np.nan)
    a, b, i, j = np.unravel_index(np.nanargmax(T), T.shape)
    return T[a, b, i, j], MUS[a], SIGMAS[b], extent[0] + (i + 0.5) * pix, extent[2] + (j + 0.5) * pix

## 5.2 Three SDSS satellites

Each patch is a circle (2° radius for Boötes I, 1.5° for Willman 1 and Segue 1). We search the inner region, at
least 0.5° from the edge, so that the 30′ background kernel stays within the data.

In [ ]:
patches = {'bootes1_sdss': 'bootes_1', 'willman1_sdss': 'willman_1', 'segue1_sdss': 'segue_1'}
radius = {'bootes1_sdss': 2.0, 'willman1_sdss': 1.5, 'segue1_sdss': 1.5}
results = {}
for name, key in patches.items():
    ref = lvdb.loc[key]
    cat = load(name)
    x, y = tangent(cat.ra.to_numpy(), cat.dec.to_numpy(), ref.ra, ref.dec)
    g0 = (cat.psfmag_g - cat.extinction_g).to_numpy()
    c0 = g0 - (cat.psfmag_r - cat.extinction_r).to_numpy()
    rmax = radius[name] - 0.02
    extent = (-rmax, rmax, -rmax, rmax)
    pc = np.arange(-rmax, rmax, 1 / 60) + 1 / 120
    PX, PY = np.meshgrid(pc, pc, indexing='ij')
    mask = np.hypot(PX, PY) < rmax
    inner = np.hypot(PX, PY) < rmax - 0.5
    w = cmd_weights(c0, g0)
    S = search(x, y, w, mask, extent)
    results[name] = dict(x=x, y=y, c0=c0, g0=g0, w=w, S=S, mask=mask, inner=inner, extent=extent, PX=PX, PY=PY,
                         peak=peak(S, extent, inner=inner))
    smax, mu, sig, px, py = results[name]['peak']
    off = np.hypot(px, py) * 60
    print(f'{key:10s}: max S = {smax:5.1f} at mu = {mu:4.1f}, sigma = {sig}′, {off:4.1f}′ from the LVDB centre '
          f'(LVDB: mu = {ref.distance_modulus:.2f}, r_h = {ref.rhalf:.1f}′)')

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
for a, (name, r) in zip(ax, results.items()):
    smax, mu, sig, px, py = r['peak']
    a_mu, a_sig = list(MUS).index(mu), list(SIGMAS).index(sig)
    im = a.imshow(r['S'][a_mu, a_sig].T, origin='lower', extent=r['extent'], cmap='Reds', vmin=0, vmax=8)
    a.contour(r['PX'], r['PY'], r['inner'], [0.5], colors='k', linewidths=0.6)
    a.set(title=f'{patches[name]}: μ = {mu}, σ = {sig}′, max S = {smax:.1f}', xlabel=r'$\xi$ [deg]', ylabel=r'$\eta$ [deg]')
    a.invert_xaxis()
    plt.colorbar(im, ax=a, label='S')
plt.tight_layout()

**Figure 5.1.** Significance maps at the (μ, σ) of the maximum, for each patch (negative fluctuations are set to 0);
the black circle bounds the searched region. Each satellite is the highest peak of its patch, close to its catalogued position, and the best μ is close to
its catalogued distance modulus (on our 0.5-mag grid).

Is S = 10 a lot? $S$ is roughly Gaussian *at one pixel, one kernel and one μ*. But we searched ~10⁴ pixels (with
overlapping kernels), 4 kernels and 11 distances: the largest of many correlated Gaussian-ish numbers is typically
4–5, not 0. To know what to expect without a satellite, we must simulate the search.

## 5.3 Calibrating the maximum with null patches

A **null patch** keeps every star's magnitudes and colours (so the CMD and the weights are unchanged) but places the
stars at random, uniformly over the observed area — destroying any clustering. We run the identical search on 200
null patches per field, record the maximum of $S$ over the inner region, all kernels and all μ, and compare.

(This null is optimistic in one way: the real sky has smooth gradients in star counts, which uniform nulls lack; the
30′ background kernel absorbs most of them. A stricter null would use real fields without known satellites.)

In [ ]:
rng = np.random.default_rng(5)
n_null = 200
null_max = {}
for name, r in results.items():
    rmax = r['extent'][1]
    n = len(r['x'])
    m = []
    for t in range(n_null):
        rr, th = rmax * np.sqrt(rng.uniform(0, 1, n)), rng.uniform(0, 2 * np.pi, n)   # uniform in the circle
        Sn = search(rr * np.cos(th), rr * np.sin(th), r['w'], r['mask'], r['extent'])
        m.append(np.nanmax(np.where(r['inner'][None, None], Sn, np.nan)))
    null_max[name] = np.array(m)
    p99 = np.percentile(null_max[name], 99)
    print(f'{patches[name]:10s}: null maxima median {np.median(null_max[name]):.2f}, 99th percentile {p99:.2f}; '
          f'observed {r["peak"][0]:.1f}')

fig, ax = plt.subplots(figsize=(7, 4))
for name, v in null_max.items():
    ax.hist(v, np.linspace(2, 9, 36), histtype='step', lw=2, label=f'null maxima, {patches[name]} patch')
    ax.axvline(min(results[name]['peak'][0], 8.9), ls='--', c=ax.lines[-1].get_color() if ax.lines else 'k')
ax.set(xlabel='maximum S over the whole search', ylabel='null patches')
ax.legend(fontsize=8)
plt.tight_layout()

for name, key in patches.items():
    ref = lvdb.loc[key]
    smax, mu, sig, px, py = results[name]['peak']
    tol = max(1 / 60, 0.5 * ref.rhalf / 60)                      # one pixel or half the half-light radius
    assert np.hypot(px, py) < tol, (key, np.hypot(px, py) * 60)
    assert smax > np.percentile(null_max[name], 99), key

**Figure 5.2.** Distribution of the maximum of $S$ over the whole search for 200 null patches of each field (solid);
the observed maxima (dashed lines; drawn at the edge of the plot when they lie beyond it). Without a satellite the
search still returns maxima of ~4.5–5.5. All three satellites lie far above the 99th percentile of their nulls.

With 200 null patches the empirical tail is known only to probability ~1/200; a claim like "p < 10⁻⁶" would need
millions of nulls or a model of the tail. The practical lesson: **set the detection threshold from the null
distribution of the maximum, not from the Gaussian tail of a single test.**

## Summary

- Weighted star maps, smoothed with a small and a large kernel, give a significance map $S$ whose single-pixel
  distribution is close to Gaussian.
- Scanning sizes and distances is necessary (satellites span a wide range) but inflates the maximum.
- Null patches calibrate the maximum over the whole search; Boötes I, Willman 1 and Segue 1 are each the highest
  peak of their patch, at their catalogued positions, far above the null maxima.